In [1]:
from repeatradar import *
import pandas as pd

# Testing Improved repeatradar Package

This notebook demonstrates the new and improved `generate_cohort_data` function with:

1. **Better parameter names** (more user-friendly)
2. **Period duration as strings** ('D', 'W', 'M', 'Q', 'Y')
3. **Missing period handling** (complete triangles with 0 values)
4. **Retention rate calculations** (percentage compared to period 0)
5. **Enhanced functionality** while maintaining performance

In [2]:
ecommerce_data = pd.read_pickle("https://github.com/krinya/repeatradar/raw/refs/heads/main/examples/data/ecommerce_data_1.pkl")

In [3]:
generated_data_pivot_user = generate_cohort_data(
    data = ecommerce_data,
    datetime_column_name = 'InvoiceDateTime',
    user_column_name = 'CustomerID',
    base_period = 'M',
    period_duration = 30,
    output_format = 'pivot'
)

In [4]:
generated_data_pivot_user = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',      # Previously: datetime_column_name
    user_column='CustomerID',           # Previously: user_column_name
    cohort_period='M',                  # Previously: base_period
    period_duration=30,                 # Can now also be 'M', 'W', etc.
    output_format='pivot'
)

print(f"User cohort data shape: {generated_data_pivot_user.shape}")
generated_data_pivot_user

TypeError: generate_cohort_data() got an unexpected keyword argument 'date_column'

## 🆕 New Feature: Period Duration as Strings

Now you can use period strings instead of calculating days manually!

In [ ]:
# Weekly analysis periods (NEW FEATURE!)
weekly_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    period_duration='W',  # Weekly periods - much easier than period_duration=7!
    cohort_period='M'     # Monthly cohorts
)

print(f"Weekly analysis shape: {weekly_cohorts.shape}")
print("\nFirst 5 rows and columns:")
weekly_cohorts.iloc[:5, :5]

In [ ]:
# Compare different period durations
print("Comparison of different period durations:")
print(f"Daily periods ('D'):     {generate_cohort_data(data=ecommerce_data, date_column='InvoiceDateTime', user_column='CustomerID', period_duration='D').shape}")
print(f"Weekly periods ('W'):    {generate_cohort_data(data=ecommerce_data, date_column='InvoiceDateTime', user_column='CustomerID', period_duration='W').shape}")
print(f"Monthly periods ('M'):   {generate_cohort_data(data=ecommerce_data, date_column='InvoiceDateTime', user_column='CustomerID', period_duration='M').shape}")
print(f"Quarterly periods ('Q'): {generate_cohort_data(data=ecommerce_data, date_column='InvoiceDateTime', user_column='CustomerID', period_duration='Q').shape}")
print(f"Yearly periods ('Y'):    {generate_cohort_data(data=ecommerce_data, date_column='InvoiceDateTime', user_column='CustomerID', period_duration='Y').shape}")

## 🆕 New Feature: Retention Rate Calculations

Calculate retention rates as percentages compared to the acquisition period (period 0)!

In [ ]:
# Retention rate analysis (NEW FEATURE!)
retention_rates = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    calculate_retention_rate=True,  # NEW: Calculate retention as percentage!
    period_duration='M'              # Monthly periods for better visibility
)

print(f"Retention rates shape: {retention_rates.shape}")
print("\nRetention rates (% of users returning compared to acquisition period):")
retention_rates.round(1)  # Round to 1 decimal place for better readability

## 🆕 Improved: Missing Period Handling

All periods are now included with 0 values where data is missing - no more gaps in your cohort triangles!

In [ ]:
# Long format showing complete periods (IMPROVED!)
long_format = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    output_format='long',
    period_duration='M'
)

print(f"Total rows in long format: {len(long_format)}")
print("\nSample of data including periods with 0 values:")
print(long_format.head(15))

print("\nPeriods with 0 values (previously these would be missing):")
zero_periods = long_format[long_format['metric_value'] == 0]
print(f"Found {len(zero_periods)} periods with 0 values")
print(zero_periods.head(10))

## Revenue Cohort Analysis with New Features

Demonstrate value-based cohorts using the improved function.

In [ ]:
# Revenue cohort analysis with weekly periods
revenue_cohorts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',      # Previously: value_column_name
    aggregation_function='sum',
    period_duration='W',            # Weekly revenue analysis
    cohort_period='M'               # Monthly cohorts
)

print(f"Revenue cohorts shape: {revenue_cohorts.shape}")
print("\nRevenue by cohort and week (first 5 cohorts and 8 weeks):")
revenue_cohorts.iloc[:5, :8].round(2)

## Advanced Examples

More sophisticated analyses using the new features.

In [ ]:
# Advanced: Multiple analysis types comparison
print("=== ADVANCED ANALYSIS COMPARISON ===")

# 1. User count analysis
user_counts = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    period_duration='M'
)

# 2. Average order value analysis
avg_order_value = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',
    aggregation_function='mean',
    period_duration='M'
)

# 3. Total revenue analysis
total_revenue = generate_cohort_data(
    data=ecommerce_data,
    date_column='InvoiceDateTime',
    user_column='CustomerID',
    value_column='TotalPrice',
    aggregation_function='sum',
    period_duration='M'
)

print(f"User counts shape: {user_counts.shape}")
print(f"Avg order value shape: {avg_order_value.shape}")
print(f"Total revenue shape: {total_revenue.shape}")

print("\n--- Period 0 comparison (acquisition period) ---")
print(f"Users acquired in first cohort: {user_counts.iloc[0, 0]}")
print(f"Avg order value in first cohort: ${avg_order_value.iloc[0, 0]:.2f}")
print(f"Total revenue in first cohort: ${total_revenue.iloc[0, 0]:.2f}")

## Error Handling & Validation

Demonstrate improved error messages and validation.

In [ ]:
# Demonstrate improved error handling
print("=== ERROR HANDLING EXAMPLES ===")

# 1. Try retention rate with value column (should fail)
try:
    generate_cohort_data(
        data=ecommerce_data,
        date_column='InvoiceDateTime',
        user_column='CustomerID',
        value_column='TotalPrice',
        calculate_retention_rate=True  # This should fail!
    )
except ValueError as e:
    print(f"✓ Caught expected error: {e}")

# 2. Try with non-existent column (should fail with helpful message)
try:
    generate_cohort_data(
        data=ecommerce_data,
        date_column='NonExistentColumn',
        user_column='CustomerID'
    )
except ValueError as e:
    print(f"✓ Caught expected error: {e}")

print("\n✓ All error handling working correctly!")

## Summary of Improvements

### ✅ What's New:
1. **Better parameter names**: `date_column`, `user_column`, `value_column`, `cohort_period`
2. **String period durations**: Use `'D'`, `'W'`, `'M'`, `'Q'`, `'Y'` instead of calculating days
3. **Missing period handling**: Complete triangles with 0 values where data is missing
4. **Retention rate calculations**: Automatic percentage calculations vs. acquisition period
5. **Enhanced validation**: Better error messages and input validation
6. **Maintained performance**: All optimizations from the original function preserved

### 🔄 Backward Compatibility:
- Old parameter names still work (but are deprecated)
- All existing functionality preserved
- Same performance characteristics
- Same output formats supported

In [ ]:
# Final demonstration: Side-by-side comparison
print("=== FINAL COMPARISON: OLD vs NEW SYNTAX ===")
print("\nOLD (deprecated but still works):")
print("generate_cohort_data(data, datetime_column_name='date', user_column_name='user', base_period='M', period_duration=30)")

print("\nNEW (recommended):")
print("generate_cohort_data(data, date_column='date', user_column='user', cohort_period='M', period_duration='M')")

print("\n🎉 The function is now more intuitive, powerful, and user-friendly!")